In [1]:
!pip install spotipy==2.25.2
!pip install ftfy==6.3.1


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.6 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Standard library imports
import os
import random
import time
from collections import deque

# Third-party imports
import pandas as pd
import requests
import spotipy
from ftfy import fix_text
from spotipy.exceptions import SpotifyException
from spotipy.oauth2 import SpotifyClientCredentials

In [3]:
# Import the CSV from the last stage of the pipeline

filepath = '/work/pipeline/3.7.Wide_composer_analysis.csv'
df = pd.read_csv(filepath)
df.head()

/tmp/ipykernel_576/2913811196.py:4: DtypeWarning: Columns (45,49,58,84) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath)


,tmdb_id,film_title,film_adult,film_runtime_min,film_genres,film_rating,film_vote_count,film_mpaa_rating,film_original_title,film_language_name,...,film_is_music,film_is_mystery,film_is_romance,film_is_science_fiction,film_is_tv_movie,film_is_thriller,film_is_war,film_is_western,vote_count_above_500,composer_primary_clean
0,1027160,Alone in the Dark,False,90,"Horror, Thriller",4.769,13,NR,Alone in the Dark,English,...,False,False,False,False,False,True,False,False,False,Christopher French
1,1027160,Alone in the Dark,False,90,"Horror, Thriller",4.769,13,NR,Alone in the Dark,English,...,False,False,False,False,False,True,False,False,False,Christopher French
2,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,False,False,False,False,False,False,False,False,False,Tim Dewit
3,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,False,False,False,False,False,False,False,False,False,Tim Dewit
4,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,False,False,False,False,False,False,False,False,False,Tim Dewit


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78992 entries, 0 to 78991
Columns: 131 entries, tmdb_id to composer_primary_clean
dtypes: bool(29), float64(15), int64(18), object(69)
memory usage: 63.7+ MB


In [5]:
# Create a variable to store a subset of the columns for convenience

columns = [
    'recording_artist_credit',
    'track_title',
    'album_title',
    'tmdb_id',
    'film_title',
    'film_revenue',
    'film_genres',
    'film_imdb_id',
    'release_group_mbid',
    'release_mbid',
    'rg_primary_type',
    'rg_secondary_types',
    'album_soundtrack_type',
    'is_canonical_soundtrack',
    'is_canonical_songtrack',
    'album_artist_mbids_text',
    'album_artist_names_text',
    'album_artist_types_text',
    'film_soundtrack_composer_raw',
    'composer_names_text',
    'track_number',
    'track_length_ms',
    'recording_mbid',
    'isrcs_text',
    'vote_count_above_500'
    ]

In [6]:
# Clean text for instances where apostrophe displays as extraneous symbols

df["track_title_cleaned"] = (
    df["track_title"]
    .astype(str)
    .map(fix_text)
    .str.replace("’", "'", regex=False)
)

# Spotify API Code

The Spotify API was queried in a local environment \(PyCharm\) in two separate runs\. The code below is for reference only and should not be re\-run\. The results are saved in the api\_dumps folder on Deepnote and will be merged into the wider dataframe\.

In [7]:
# # Load credentials
# SPOTIPY_CLIENT_ID = os.getenv("SPOTIFY_CLIENT_ID")
# SPOTIPY_CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")

In [8]:
# #------------------------
# # Spotipy client (one-time)
# #------------------------

# def make_spotify_client(client_id=None, client_secret=None):
#     """
#     If client_id/client_secret are None, Spotipy reads from:
#       SPOTIPY_CLIENT_ID, SPOTIPY_CLIENT_SECRET
#     """
#     auth_manager = SpotifyClientCredentials(
#         client_id=(client_id.strip() if isinstance(client_id, str) else None),
#         client_secret=(client_secret.strip() if isinstance(client_secret, str) else None),
#     )
#     return spotipy.Spotify(auth_manager=auth_manager, requests_timeout=30, retries=0)

# #------------------------
# # Throttling helpers
# #------------------------

# def throttle(base_delay=0.12, jitter=0.10):
#     """
#     Sleep a little between requests to reduce rate-limit hits.
#     base_delay: fixed seconds
#     jitter: random seconds added [0, jitter]
#     """
#     time.sleep(base_delay + random.random() * jitter)


# def backoff_from_retry_after(exc: SpotifyException, default_wait=2.0, extra_jitter=0.25):
#     retry_after = None

#     try:
#         hdrs = getattr(exc, "headers", None) or {}
#         retry_after = hdrs.get("Retry-After") or hdrs.get("retry-after")
#     except Exception:
#         retry_after = None

#     try:
#         wait_s = float(retry_after) if retry_after is not None else float(default_wait)
#     except Exception:
#         wait_s = float(default_wait)

#     # sleep happens outside now
#     return wait_s + random.random() * extra_jitter


# #------------------------
# # Helper function to keep queries safely under Spotify's ~250 character limit
# #------------------------

# def safe_text(s, max_len=200):
#     return (s or "").strip()[:max_len]

# #------------------------
# # simple in-memory cache (define this once, above the function)
# #------------------------

# spotify_cache = {}

# #------------------------
# # Track lookup helper function
# # -------------------------

# def lookup_track_id(sp, track_title: str, artist_name: str, market=None, limit=3):
#     track_title = safe_text(track_title, 200)
#     artist_name = safe_text(artist_name, 200)

#     if not track_title or not artist_name:
#         return None

#     # Cache key (case-insensitive)
#     cache_key = (track_title.lower(), artist_name.lower())

#     # Return cached result if we’ve seen this before
#     if cache_key in spotify_cache:
#         return spotify_cache[cache_key]

#     q = f'track:"{track_title}" artist:"{artist_name}"'

#     results = sp.search(q=q, type="track", limit=limit, market=market)
#     items = results.get("tracks", {}).get("items", [])

#     if not items:
#         spotify_cache[cache_key] = None   # cache the miss
#         return None

#     t = items[0]
#     result = {
#         "spotify_track_id": t.get("id"),
#         "spotify_url": (t.get("external_urls") or {}).get("spotify"),
#         "matched_track_name": t.get("name"),
#         "matched_artists": ", ".join([a.get("name", "") for a in t.get("artists", [])]),
#         "matched_album": (t.get("album") or {}).get("name"),
#         "matched_release_date": (t.get("album") or {}).get("release_date"),
#         "matched_popularity": t.get("popularity"),
#         "query_used": q,
#     }

#     # Store successful result in cache
#     spotify_cache[cache_key] = result
#     return result

# #------------------------
# # Bulk lookup function
# #------------------------

# def bulk_lookup_spotify_ids(
#     df_in: pd.DataFrame,
#     client_id=None,
#     client_secret=None,
#     base_delay=0.12,
#     jitter=0.10,
#     max_attempts=3,
#     market=None,
#     batch_size=5000,          # size of each chunk
#     start_row=0,              # resume from this row index (0-based)
#     checkpoint_path=None,     # write progress as you go
#     checkpoint_every=500,     # how often to flush progress
# ):
#     sp = make_spotify_client(client_id=client_id, client_secret=client_secret)

#     out_rows = []
#     n_total = len(df_in)

#     # Process only one batch
#     end_row = min(start_row + batch_size, n_total)
#     df_batch = df_in.iloc[start_row:end_row].copy()

#     # Use enumerate so progress is clean even if df index isn't 0..n-1
#     for k, (_, row) in enumerate(df_batch.iterrows(), start=1):
#         artist = row.get("recording_artist_credit", "")
#         title = row.get("track_title_cleaned", "")

#         result = None
#         error = None

#         for attempt in range(1, max_attempts + 1):
#             try:
#                 throttle(base_delay=base_delay, jitter=jitter)
#                 result = lookup_track_id(sp, track_title=title, artist_name=artist, market=market)
#                 error = None
#                 break

#             except SpotifyException as exc:
#                 if exc.http_status == 429:
#                     wait_s = backoff_from_retry_after(exc, default_wait=2.0)

#                     # Hard-stop if Spotify tells us to wait "too long"
#                     if wait_s > 300:   # 5 minutes (tune if you want)
#                         print(
#                             f"\n🚨 Spotify rate limit cooldown too long ({int(wait_s)}s). "
#                             f"Checkpointing and exiting safely."
#                         )

#                         # flush anything we have
#                         if checkpoint_path and out_rows:
#                             tmp = pd.DataFrame(out_rows)
#                             header = not os.path.exists(checkpoint_path)
#                             tmp.to_csv(checkpoint_path, mode="a", header=header, index=False)
#                             out_rows.clear()

#                         raise SystemExit(
#                             f"Spotify requested long cooldown ({int(wait_s)}s). "
#                             f"Resume later."
#                         )

#                     # normal short backoff
#                     time.sleep(wait_s)
#                     error = "rate_limited_429"
#                     continue

#             except Exception as exc:
#                 time.sleep(0.75 + random.random() * 0.25)
#                 error = f"other_error_{type(exc).__name__}"
#                 continue

#         out_rows.append({
#             "sp_tracking_id": row.get("sp_tracking_id", None),
#             "recording_artist_credit": artist,
#             "track_title_cleaned": title,  # NOTE: changed key name to match your merge below
#             "spotify_track_id": (result or {}).get("spotify_track_id") if result else None,
#             "spotify_url": (result or {}).get("spotify_url") if result else None,
#             "matched_track_name": (result or {}).get("matched_track_name") if result else None,
#             "matched_artists": (result or {}).get("matched_artists") if result else None,
#             "matched_album": (result or {}).get("matched_album") if result else None,
#             "matched_release_date": (result or {}).get("matched_release_date") if result else None,
#             "matched_popularity": (result or {}).get("matched_popularity") if result else None,
#             "query_used": (result or {}).get("query_used") if result else None,
#             "lookup_error": error,
#             "attempts_used": attempt,
#         })

#         # progress within the batch + absolute progress
#         if k % 100 == 0:
#             print(f"Processed {start_row + k}/{n_total} rows...")

#         # Periodic checkpoint append
#         if checkpoint_path and (k % checkpoint_every == 0):
#             tmp = pd.DataFrame(out_rows)
#             header = not os.path.exists(checkpoint_path)
#             tmp.to_csv(checkpoint_path, mode="a", header=header, index=False)
#             out_rows.clear()  # avoid holding lots of rows in memory

#     # flush remaining rows for this batch
#     batch_df = pd.DataFrame(out_rows)
#     if checkpoint_path and len(batch_df):
#         header = not os.path.exists(checkpoint_path)
#         batch_df.to_csv(checkpoint_path, mode="a", header=header, index=False)
#         out_rows.clear()
#         return pd.DataFrame()  # results are in the checkpoint file

#     return batch_df

In [9]:
# # Loop to run in batches

# checkpoint = "scoped_spotify_id_lookup_results.csv"

# # If rerunning from scratch, delete the checkpoint first:
# # import os
# # if os.path.exists(checkpoint): os.remove(checkpoint)

# batch_size = 5000

# for start in range(0, len(df), batch_size):
#     print(f"\n=== Batch starting at row {start} ===")
#     bulk_lookup_spotify_ids(
#         df_in=df,
#         client_id=SPOTIPY_CLIENT_ID,
#         client_secret=SPOTIPY_CLIENT_SECRET,
#         base_delay=0.25, # Increased from 0.12 due to repeated 429 errors
#         jitter=0.10,
#         max_attempts=3,
#         market=None,
#         batch_size=batch_size,
#         start_row=start,
#         checkpoint_path=checkpoint,   # writes progress continuously
#         checkpoint_every=500,
#     )

# print("Checkpoint saved at:", checkpoint)

In the first Spotify run, we tried to query the full population of movies without applying the \>= 500 vote count filter\. We collected as many results as we could before hitting the rate limit several times and shifting our approach\.

In [10]:
# Read CSV from the first Spotify API run

filepath_first_run = '/work/api_dumps/pycharm_spotify_id_lookup_results.csv'
df_first_run = pd.read_csv(filepath_first_run)
df_first_run.head()

,recording_artist_credit,track_title_cleaned,spotify_track_id,spotify_url,matched_track_name,matched_artists,matched_album,matched_release_date,matched_popularity,query_used,lookup_error,attempts_used
0,Philippe Vachey,Alone in the Dark Theme,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,Philippe Vachey,[data track],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,Mr. Brady,Disqusting,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,Rob the Viking,Move It Up,21h0xKdO8E7EY0H90fApYB,https://open.spotify.com/track/21h0xKdO8E7EY0H...,Move It Up,Rob The Viking,Beats to Pillage and Conquer By,1/1/2003,1.0,"track:""Move It Up"" artist:""Rob the Viking""",NaN,1
4,Son Doobie,Party People,1ZvGInqvCALu6OxcGR4G5H,https://open.spotify.com/track/1ZvGInqvCALu6Ox...,Party People,Son Doobie,Funk Superhero,5/20/2003,6.0,"track:""Party People"" artist:""Son Doobie""",NaN,1


In [11]:
len(df_first_run)

47500

In [12]:
# Rename popularity column for convenience

df_first_run.rename(columns={"matched_popularity" : "spotify_popularity"}, inplace=True)
df_first_run.head()

,recording_artist_credit,track_title_cleaned,spotify_track_id,spotify_url,matched_track_name,matched_artists,matched_album,matched_release_date,spotify_popularity,query_used,lookup_error,attempts_used
0,Philippe Vachey,Alone in the Dark Theme,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,Philippe Vachey,[data track],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,Mr. Brady,Disqusting,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,Rob the Viking,Move It Up,21h0xKdO8E7EY0H90fApYB,https://open.spotify.com/track/21h0xKdO8E7EY0H...,Move It Up,Rob The Viking,Beats to Pillage and Conquer By,1/1/2003,1.0,"track:""Move It Up"" artist:""Rob the Viking""",NaN,1
4,Son Doobie,Party People,1ZvGInqvCALu6OxcGR4G5H,https://open.spotify.com/track/1ZvGInqvCALu6Ox...,Party People,Son Doobie,Funk Superhero,5/20/2003,6.0,"track:""Party People"" artist:""Son Doobie""",NaN,1


In [13]:
num_rows_df = len(df_first_run)
num_rows_spotify_ids = df_first_run["spotify_track_id"].count()
print(f"In the 1st run file, there are {num_rows_spotify_ids} out of {num_rows_df} total rows with spotify ID values")

In the 1st run file, there are 39431 out of 47500 total rows with spotify ID values


In [14]:
# De-duplicate on artist-track before merging (if duplicates found, keep row with lowest Spotify ID)

df_spotify_ids_deduped = (
    df_first_run
    .sort_values("spotify_track_id")  
    .drop_duplicates(
        subset=["recording_artist_credit", "track_title_cleaned"],
        keep="first"
    )
)

print(f"Number of records after de-dupe: {len(df_spotify_ids_deduped)}")
df_spotify_ids_deduped.head()

Number of records after de-dupe: 44530


,recording_artist_credit,track_title_cleaned,spotify_track_id,spotify_url,matched_track_name,matched_artists,matched_album,matched_release_date,spotify_popularity,query_used,lookup_error,attempts_used
42610,Clint Mansell,The Woods,000Kj67F3VpH5MhKzY0KW7,https://open.spotify.com/track/000Kj67F3VpH5Mh...,The Woods,Clint Mansell,In the Earth (Original Music),2021-04-16,1.0,"track:""The Woods"" artist:""Clint Mansell""",NaN,1
5005,Daniel Pemberton,It's an Abstract,002dozOGSO8AhuXoz7kDZT,https://open.spotify.com/track/002dozOGSO8AhuX...,It's An Abstract,Daniel Pemberton,Steve Jobs (Original Motion Picture Soundtrack),10/9/2015,15.0,"track:""It's an Abstract"" artist:""Daniel Pember...",NaN,1
38428,Benji Merrison,Scheduling,005EKQXOTKo9JzZfKDifyN,https://open.spotify.com/track/005EKQXOTKo9JzZ...,Scheduling,Benji Merrison,General Magic (Original Film Soundtrack),2019-01-04,2.0,"track:""Scheduling"" artist:""Benji Merrison""",NaN,1
16079,Ryuichi Sakamoto,B29,005i8ddvsWYBJRaePWAs12,https://open.spotify.com/track/005i8ddvsWYBJRa...,B29,Ryuichi Sakamoto,Nagasaki: Memories of my Son (Original Soundtr...,12/9/2015,5.0,"track:""B29"" artist:""Ryuichi Sakamoto""",NaN,1
39718,Gabriel Yared,Madame Rosa,005xfoYaMudYZUwbq5GrVc,https://open.spotify.com/track/005xfoYaMudYZUw...,Madame Rosa,Gabriel Yared,The Life Ahead (Original Motion Picture Soundt...,2020-11-13,5.0,"track:""Madame Rosa"" artist:""Gabriel Yared""",NaN,1


In [15]:
# Merging the wide dataframe with the 1st run dataframe - Left Join

first_run_merged_df = df.merge(
    df_spotify_ids_deduped[
        ["recording_artist_credit", "track_title_cleaned", "spotify_track_id", "spotify_url", "spotify_popularity"]
    ],
    on=["recording_artist_credit", "track_title_cleaned"],
    how="left"
)

print(f"Number of records: {len(first_run_merged_df)}")
first_run_merged_df.head()

Number of records: 78992


,tmdb_id,film_title,film_adult,film_runtime_min,film_genres,film_rating,film_vote_count,film_mpaa_rating,film_original_title,film_language_name,...,film_is_tv_movie,film_is_thriller,film_is_war,film_is_western,vote_count_above_500,composer_primary_clean,track_title_cleaned,spotify_track_id,spotify_url,spotify_popularity
0,1027160,Alone in the Dark,False,90,"Horror, Thriller",4.769,13,NR,Alone in the Dark,English,...,False,True,False,False,False,Christopher French,Alone in the Dark Theme,NaN,NaN,NaN
1,1027160,Alone in the Dark,False,90,"Horror, Thriller",4.769,13,NR,Alone in the Dark,English,...,False,True,False,False,False,Christopher French,[data track],NaN,NaN,NaN
2,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,False,False,False,False,False,Tim Dewit,Gloc,7kVnKJS0UVB8ZtHC4czKBv,https://open.spotify.com/track/7kVnKJS0UVB8ZtH...,2.0
3,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,False,False,False,False,False,Tim Dewit,Move It Up,21h0xKdO8E7EY0H90fApYB,https://open.spotify.com/track/21h0xKdO8E7EY0H...,1.0
4,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,False,False,False,False,False,Tim Dewit,Party People,1ZvGInqvCALu6OxcGR4G5H,https://open.spotify.com/track/1ZvGInqvCALu6Ox...,6.0


In the second Spotify run, we limited the population of movies to those with \>= 500 vote count for which we did not obtain the ID's in the first run\.

In [16]:
# For the second run, create a dataframe that only has films with vote count >= 500

df_above_500 = df[df['vote_count_above_500'] == True]

print(f"Number of records: {len(df_above_500)}")
df_above_500.head()

Number of records: 36580


,tmdb_id,film_title,film_adult,film_runtime_min,film_genres,film_rating,film_vote_count,film_mpaa_rating,film_original_title,film_language_name,...,film_is_mystery,film_is_romance,film_is_science_fiction,film_is_tv_movie,film_is_thriller,film_is_war,film_is_western,vote_count_above_500,composer_primary_clean,track_title_cleaned
66,823625,Blacklight,False,104,"Action, Thriller, Adventure",6.164,1221,PG-13,Blacklight,English,...,False,False,False,False,True,False,False,True,Mark Isham,Dragonfire - The Spacecraft
67,823625,Blacklight,False,104,"Action, Thriller, Adventure",6.164,1221,PG-13,Blacklight,English,...,False,False,False,False,True,False,False,True,Mark Isham,The Wrath of Drathro
68,823625,Blacklight,False,104,"Action, Thriller, Adventure",6.164,1221,PG-13,Blacklight,English,...,False,False,False,False,True,False,False,True,Mark Isham,Katryca (Parts One and Two)
69,823625,Blacklight,False,104,"Action, Thriller, Adventure",6.164,1221,PG-13,Blacklight,English,...,False,False,False,False,True,False,False,True,Mark Isham,The Illusion
70,823625,Blacklight,False,104,"Action, Thriller, Adventure",6.164,1221,PG-13,Blacklight,English,...,False,False,False,False,True,False,False,True,Mark Isham,Cliffhanger - Zombies - Svartos Theme


In [17]:
# Determine the number of unique track-artist pairs in the wide dataframe where vote count >= 500

grouped_df = df_above_500.groupby([
    'track_title', 
    'recording_artist_credit'
    ]).count()

grouped_df.shape[0]

36148

In [18]:
# Sort by vote_count

df_above_500 = df_above_500.sort_values(by=['film_vote_count'], ascending=False)

In [19]:
# Create a column that has a unique identifier that can be used for de-bugging calls to the Spotify API

df_above_500 = df_above_500.reset_index(drop=True)
df_above_500["sp_tracking_id"] = df_above_500.index
df_above_500.head()

,tmdb_id,film_title,film_adult,film_runtime_min,film_genres,film_rating,film_vote_count,film_mpaa_rating,film_original_title,film_language_name,...,film_is_romance,film_is_science_fiction,film_is_tv_movie,film_is_thriller,film_is_war,film_is_western,vote_count_above_500,composer_primary_clean,track_title_cleaned,sp_tracking_id
0,293660,Deadpool,False,108,"Action, Adventure, Comedy",7.623,32247,R,Deadpool,English,...,False,False,False,False,False,False,True,Tom Holkenborg,Liam Neeson Nightmares,0
1,293660,Deadpool,False,108,"Action, Adventure, Comedy",7.623,32247,R,Deadpool,English,...,False,False,False,False,False,False,True,Tom Holkenborg,Going Commando,1
2,293660,Deadpool,False,108,"Action, Adventure, Comedy",7.623,32247,R,Deadpool,English,...,False,False,False,False,False,False,True,Tom Holkenborg,Four or Five Moments,2
3,293660,Deadpool,False,108,"Action, Adventure, Comedy",7.623,32247,R,Deadpool,English,...,False,False,False,False,False,False,True,Tom Holkenborg,Every Time I See Her,3
4,293660,Deadpool,False,108,"Action, Adventure, Comedy",7.623,32247,R,Deadpool,English,...,False,False,False,False,False,False,True,Tom Holkenborg,Watership Down,4


In [20]:
# Read CSV from the second Spotify API run

filepath_second_run = '/work/api_dumps/scoped_spotify_id_lookup_results.csv'
df_spotify_second_run = pd.read_csv(filepath_second_run)

print(f"Number of records: {len(df_spotify_second_run)}")
df_spotify_second_run.head()

Number of records: 15838


,sp_tracking_id,recording_artist_credit,track_title_cleaned,spotify_track_id,spotify_url,matched_track_name,matched_artists,matched_album,matched_release_date,matched_popularity,query_used,lookup_error,attempts_used
0,3,Neil Sedaka,Calendar Girl (1999 remastered version),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,127,Tom Holkenborg aka Junkie XL,Storm Is Coming (extended version),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,128,Tom Holkenborg aka Junkie XL & Christian Vorlä...,Claw Trucks (extended version),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,129,Tom Holkenborg aka Junkie XL,The Rig (extended version),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,130,Tom Holkenborg aka Junkie XL,Brothers in Arms (extended version),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1


In [21]:
# Rename popularity column for convenience

df_spotify_second_run.rename(columns={"matched_popularity" : "spotify_popularity"}, inplace=True)
df_spotify_second_run.head()

,sp_tracking_id,recording_artist_credit,track_title_cleaned,spotify_track_id,spotify_url,matched_track_name,matched_artists,matched_album,matched_release_date,spotify_popularity,query_used,lookup_error,attempts_used
0,3,Neil Sedaka,Calendar Girl (1999 remastered version),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,127,Tom Holkenborg aka Junkie XL,Storm Is Coming (extended version),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,128,Tom Holkenborg aka Junkie XL & Christian Vorlä...,Claw Trucks (extended version),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,129,Tom Holkenborg aka Junkie XL,The Rig (extended version),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,130,Tom Holkenborg aka Junkie XL,Brothers in Arms (extended version),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1


In [22]:
# De-duplicate on artist-track before merging (if duplicates found, keep row with lowest Spotify ID)

df_spotify_ids_deduped = (
    df_spotify_second_run
    .sort_values("spotify_track_id")  
    .drop_duplicates(
        subset=["recording_artist_credit", "track_title_cleaned"],
        keep="first"
    )
)

In [23]:
# Merge the wide file (with first run IDs) with the second run (left join)

second_run_merged_df = first_run_merged_df.merge(
    df_spotify_ids_deduped[
        ["recording_artist_credit", "track_title_cleaned", "spotify_track_id", "spotify_url", "spotify_popularity"]
    ],
    on=["recording_artist_credit", "track_title_cleaned"],
    how="left"
)

print(f"Number of records after merge: {len(first_run_merged_df)}")
second_run_merged_df.head()

Number of records after merge: 78992


,tmdb_id,film_title,film_adult,film_runtime_min,film_genres,film_rating,film_vote_count,film_mpaa_rating,film_original_title,film_language_name,...,film_is_western,vote_count_above_500,composer_primary_clean,track_title_cleaned,spotify_track_id_x,spotify_url_x,spotify_popularity_x,spotify_track_id_y,spotify_url_y,spotify_popularity_y
0,1027160,Alone in the Dark,False,90,"Horror, Thriller",4.769,13,NR,Alone in the Dark,English,...,False,False,Christopher French,Alone in the Dark Theme,NaN,NaN,NaN,NaN,NaN,NaN
1,1027160,Alone in the Dark,False,90,"Horror, Thriller",4.769,13,NR,Alone in the Dark,English,...,False,False,Christopher French,[data track],NaN,NaN,NaN,NaN,NaN,NaN
2,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,False,False,Tim Dewit,Gloc,7kVnKJS0UVB8ZtHC4czKBv,https://open.spotify.com/track/7kVnKJS0UVB8ZtH...,2.0,NaN,NaN,NaN
3,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,False,False,Tim Dewit,Move It Up,21h0xKdO8E7EY0H90fApYB,https://open.spotify.com/track/21h0xKdO8E7EY0H...,1.0,NaN,NaN,NaN
4,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,False,False,Tim Dewit,Party People,1ZvGInqvCALu6OxcGR4G5H,https://open.spotify.com/track/1ZvGInqvCALu6Ox...,6.0,NaN,NaN,NaN


In [24]:
# Combine the _x and _y columns after merge

second_run_merged_df["spotify_track_id"] = second_run_merged_df["spotify_track_id_y"].combine_first(
    second_run_merged_df["spotify_track_id_x"]
)

second_run_merged_df["spotify_url"] = (
    second_run_merged_df["spotify_url_y"]
    .combine_first(second_run_merged_df["spotify_url_x"])
)

second_run_merged_df["spotify_popularity"] = (
    second_run_merged_df["spotify_popularity_y"]
    .combine_first(second_run_merged_df["spotify_popularity_x"])
)

In [25]:
# Drop the old columns

second_run_merged_df = second_run_merged_df.drop(
    columns=[
        "spotify_track_id_x",
        "spotify_track_id_y",
        "spotify_url_x",
        "spotify_url_y",
        "spotify_popularity_x",
        "spotify_popularity_y",
    ],
    errors="ignore"
)


In [26]:
second_run_merged_df.head()

,tmdb_id,film_title,film_adult,film_runtime_min,film_genres,film_rating,film_vote_count,film_mpaa_rating,film_original_title,film_language_name,...,film_is_tv_movie,film_is_thriller,film_is_war,film_is_western,vote_count_above_500,composer_primary_clean,track_title_cleaned,spotify_track_id,spotify_url,spotify_popularity
0,1027160,Alone in the Dark,False,90,"Horror, Thriller",4.769,13,NR,Alone in the Dark,English,...,False,True,False,False,False,Christopher French,Alone in the Dark Theme,NaN,NaN,NaN
1,1027160,Alone in the Dark,False,90,"Horror, Thriller",4.769,13,NR,Alone in the Dark,English,...,False,True,False,False,False,Christopher French,[data track],NaN,NaN,NaN
2,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,False,False,False,False,False,Tim Dewit,Gloc,7kVnKJS0UVB8ZtHC4czKBv,https://open.spotify.com/track/7kVnKJS0UVB8ZtH...,2.0
3,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,False,False,False,False,False,Tim Dewit,Move It Up,21h0xKdO8E7EY0H90fApYB,https://open.spotify.com/track/21h0xKdO8E7EY0H...,1.0
4,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,False,False,False,False,False,Tim Dewit,Party People,1ZvGInqvCALu6OxcGR4G5H,https://open.spotify.com/track/1ZvGInqvCALu6Ox...,6.0


In [27]:
num_rows_df = len(second_run_merged_df)
num_rows_spotify_ids = second_run_merged_df["spotify_track_id"].count()
print(f"After merging both runs into the wide dataframe, there are {num_rows_spotify_ids} out of {num_rows_df} total rows with spotify ID values")

After merging both runs into the wide dataframe, there are 49931 out of 78992 total rows with spotify ID values


In [28]:
# Save wide file to CSV

out_path = "/work/pipeline/4.3.Wide_spotify_ids.csv"

second_run_merged_df.to_csv(out_path, index=False)

In [29]:
# Check track file

filepath = '/work/pipeline/3.7.Tracks_composer_analysis.csv'
df_track = pd.read_csv(filepath)

print(f"Number of records in track file before merge: {len(df_track)}")
df_track.head()

Number of records in track file before merge: 78992


,tmdb_id,film_title,release_group_id,release_id,match_method,album_us_release_date,us_date_has_missing_month,us_date_has_missing_day,medium_id,disc_number,...,recording_artist_credit,recording_first_release_year,recording_first_release_month,recording_first_release_day,isrcs_text,recording_tags_text,work_ids_text,work_titles_text,composer_names_text,lyricist_names_text
0,158852,Tomorrowland,1527294,1608979,imdb_exact,2015-06-09,False,False,1692109,1,...,Michael Giacchino,NaN,NaN,NaN,USWD11570549,NaN,14179222,Frank Frank,Michael Giacchino,NaN
1,545836,Amundsen,2122978,2367092,imdb_exact,NaN,NaN,NaN,2563555,1,...,Johan Söderqvist,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,524434,Eternals,2804779,3233921,imdb_exact,NaN,NaN,NaN,3523483,1,...,Ramin Djawadi,NaN,NaN,NaN,USHR12142555,NaN,14265689,Somewhere in Time,Ramin Djawadi,NaN
3,291851,Field of Lost Shoes,1414263,1474985,imdb_exact,2014-09-09,False,False,1540034,1,...,Frederik Wiedmann,NaN,NaN,NaN,NaN,NaN,14876001,The Language of the Winners,Frederik Wiedmann,NaN
4,419916,Funny Cow,1976129,2181386,title_contains_strict,NaN,NaN,NaN,2352738,1,...,Richard Hawley,NaN,NaN,NaN,UKPDC1800007,NaN,NaN,NaN,NaN,NaN


In [30]:
# De-duplicate on artist-track before merging (if duplicates found, keep row with lowest Spotify ID)

df_spotify_ids_deduped = (
    second_run_merged_df
    .sort_values("spotify_track_id")  
    .drop_duplicates(
        subset=["recording_artist_credit", "track_title_cleaned"],
        keep="first"
    )
)

print(f"Number of records after de-dupe: {len(df_spotify_ids_deduped)}")

Number of records after de-dupe: 78171


In [31]:
# Merge track file with the Spotify ID's from wide file (left join)

df_track_with_ids = df_track.merge(
    df_spotify_ids_deduped[[
        "recording_artist_credit", 
        "track_title", 
        "track_title_cleaned", 
        "spotify_track_id", 
        "spotify_url", 
        "spotify_popularity"
        ]],
    on=["recording_artist_credit", "track_title"],
    how="left"
)

print(f"Number of records after merge: {len(df_track_with_ids)}")
df_track_with_ids.head()

Number of records after merge: 78992


,tmdb_id,film_title,release_group_id,release_id,match_method,album_us_release_date,us_date_has_missing_month,us_date_has_missing_day,medium_id,disc_number,...,isrcs_text,recording_tags_text,work_ids_text,work_titles_text,composer_names_text,lyricist_names_text,track_title_cleaned,spotify_track_id,spotify_url,spotify_popularity
0,158852,Tomorrowland,1527294,1608979,imdb_exact,2015-06-09,False,False,1692109,1,...,USWD11570549,NaN,14179222,Frank Frank,Michael Giacchino,NaN,Frank Frank,1xHD9H5zKIZZnrYVhI013z,https://open.spotify.com/track/1xHD9H5zKIZZnrY...,3.0
1,545836,Amundsen,2122978,2367092,imdb_exact,NaN,NaN,NaN,2563555,1,...,NaN,NaN,NaN,NaN,NaN,NaN,Planning the Expedition,7qQ03CeoqvvOQN5KO2jOrx,https://open.spotify.com/track/7qQ03CeoqvvOQN5...,3.0
2,524434,Eternals,2804779,3233921,imdb_exact,NaN,NaN,NaN,3523483,1,...,USHR12142555,NaN,14265689,Somewhere in Time,Ramin Djawadi,NaN,Somewhere in Time,1hXr4J2zeXIEpuYhds0o7L,https://open.spotify.com/track/1hXr4J2zeXIEpuY...,14.0
3,291851,Field of Lost Shoes,1414263,1474985,imdb_exact,2014-09-09,False,False,1540034,1,...,NaN,NaN,14876001,The Language of the Winners,Frederik Wiedmann,NaN,The Language of the Winners,NaN,NaN,NaN
4,419916,Funny Cow,1976129,2181386,title_contains_strict,NaN,NaN,NaN,2352738,1,...,UKPDC1800007,NaN,NaN,NaN,NaN,NaN,Leaving Mikes House,2N03m8NTjSQEUe4LpTT0C9,https://open.spotify.com/track/2N03m8NTjSQEUe4...,1.0


In [32]:
# Save track file to CSV

out_path = "/work/pipeline/4.3.Tracks_spotify_ids.csv"

df_track_with_ids.to_csv(out_path, index=False)

In [33]:
# Albums and Artists carry over

import shutil

shutil.copy(
    "./pipeline/3.7.Albums_composer_analysis.csv",
    "./pipeline/4.3.Albums_spotify_ids.csv"
)

shutil.copy(
    "./pipeline/3.7.Artists_composer_analysis.csv",
    "./pipeline/4.3.Artists_spotify_ids.csv"
)

'./pipeline/4.3.Artists_spotify_ids.csv'

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b124131d-2024-4253-bb46-8043aed4b78f' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>